## Instalando as bibliotecas necessárias 

### conda install -c conda-forge tiktoken nltk

In [52]:
# Atualizo o pip caso haja uma nova versão do pip
!pip install --upgrade pip
# jsonlines e tqdm precisam ser instaladas, pois não fazem parte da biblioteca padrão do Python.
!pip install jsonlines tqdm
!conda install scikit-learn -y


Channels:
 - nvidia
 - defaults
 - conda-forge
Platform: linux-64
Solving environment: done

# All requested packages already installed.



In [53]:
# Importando as bibliotecas necessárias
# json e jsonlines: Para manipulação de arquivos JSON e JSONL (JSON por linha).
import json
import jsonlines
import tiktoken

# random: Para realizar a divisão aleatória dos dados.
import random
# Path: Facilita a manipulação de diretórios e arquivos.
from pathlib import Path
# tqdm: Adiciona uma barra de progresso durante o processamento.
from tqdm import tqdm

# Configurando a importação do tokenizador customizado
import os
import sys
sys.path.insert(0, os.path.expanduser("~/projetos/unb-ppgi0119/atividade_01"))

In [54]:
# Definindo o caminho para o arquivo de entrada
input_file = os.path.expanduser("~/projetos/unb-ppgi0119/atividade_01/corpus_completo.jsonl")

# Leitura e exibição das primeiras 10 linhas
with jsonlines.open(input_file, "r") as reader:
    # Converte reader em um iterador e lê as primeiras 10 linhas
    first_10_lines = [next(iter(reader)) for _ in range(10)]
    
# Exibindo as 10 primeiras linhas
for i, line in enumerate(first_10_lines, start=1):
    print(f"Linha {i}: {line}")

Linha 1: {'id': '81474', 'text': 'Pradamano é uma comuna italiana da região do Friuli-Venezia Giulia, província de Udine, com cerca de 2.972 habitantes. Estende-se por uma área de 16 km², tendo uma densidade populacional de 185 hab/km². Faz fronteira com Buttrio, Pavia di Udine, Premariacco, Remanzacco, Udine. ==Demografia== Categoria:Comunas de Údine (província)', 'title': 'Pradamano'}
Linha 2: {'id': '16592', 'text': 'Maria Valentina dos Anjos Costa - Doné Runhó ou Mãe Ruinhó - (Salvador, 1877 - 27 de dezembro de 1975) - Doné do Terreiro do Bogum, iniciada para o vodum Sobô. Passou para o grau de sacerdotisa aos 21 anos de idade, e assumiu a direção do terreiro após a morte da Doné Romana de Possú em 1925, porém, há alguns membros antigos do Terreiro do Bogum que discordam que Romaninha de Possu tenha dirigido o templo, onde a Doné Runhó permaneceu na direção por 50 anos. Foi sucedida por Mãe Gamo, Evangelista dos Anjos Costa de 64 anos (nasceu em 27 de dezembro de 1911) que era sua 

## 1. Preparação dos Dados
### O primeiro passo é segmentar o conjunto de dados usando o tokenizador implementado anteriormente, dividir o conjunto em treino e teste, e realizar a tokenização.

In [55]:
import os
import random
import nltk
from nltk.util import bigrams
from collections import defaultdict, Counter
from tqdm import tqdm
import jsonlines
from sklearn.model_selection import train_test_split

# Baixe os recursos necessários do NLTK
nltk.download('punkt')

# Caminho do arquivo de dados
data_file = os.path.expanduser("~/projetos/unb-ppgi0119/atividade_01/train_data.jsonl")

# Função para carregar e tokenizar os dados
def load_and_tokenize_data(data_file):
    all_text = []
    with jsonlines.open(data_file, "r") as reader:
        for item in reader:
            if "text" in item:
                all_text.append(item["text"])
    
    # Concatena todo o texto e realiza a tokenização
    full_text = " ".join(all_text)
    tokens = nltk.word_tokenize(full_text)
    return tokens

# Carregue e tokeniza os dados
tokens = load_and_tokenize_data(data_file)

# Divida os dados em treino (80%) e teste (20%) de forma aleatória
train_tokens, test_tokens = train_test_split(tokens, test_size=0.2, random_state=42)
print("Dados preparados!")


[nltk_data] Downloading package punkt to /home/eprioli/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


Dados preparados!


## 2. Implementação do Cálculo da Perplexidade e Geração de Texto
### O arquivo **bigram_model_perplexity.py** terá todas as partes necessárias: carregar os dados, tokenizar, construir o modelo de bigrama, calcular a perplexidade e gerar texto.
### Explicação do Arquivo **bigram_model_perplexity.py**
- **Carregamento e Tokenização**: A função load_and_tokenize_data carrega e tokeniza o texto do arquivo JSONL.
- **Divisão em Treino e Teste**: O código usa train_test_split para dividir os tokens em conjuntos de treino e teste.
- **Modelo de Bigrama**: A função build_bigram_model cria o modelo de bigrama, calculando as probabilidades condicionais para cada par de palavras.
- **Cálculo da Perplexidade**: A função calculate_perplexity mede a capacidade do modelo de prever o próximo token.
- **Geração de Texto**: A função generate_text cria uma sequência de palavras baseada no modelo de bigrama, começando com uma palavra inicial aleatória.

In [56]:
# Importando o arquivo bigram_model_perplexity.py
import os
import sys
sys.path.insert(0, os.path.expanduser("~/projetos/unb-ppgi0119/atividade_01"))
from bigram_model_perplexity import build_bigram_model, calculate_perplexity

# Construindo o modelo de bigrama com os tokens de treino
bigram_model = build_bigram_model(train_tokens)

# Calculando a perplexidade no conjunto de teste
perplexity = calculate_perplexity(bigram_model, test_tokens)
print("Perplexidade do modelo:", perplexity)


Perplexidade do modelo: 19638.505299772565


In [57]:
import os
import sys
import random
# Adiciona o caminho do projeto ao sistema de pesquisa de módulos
project_path = os.path.expanduser("~/projetos/unb-ppgi0119/atividade_01")
sys.path.insert(0, project_path)

# Importa todas as funções do arquivo bigram_model_perplexity.py
from bigram_model_perplexity import *

# Constrói o modelo de bigrama com os tokens de treino
bigram_model = build_bigram_model(train_tokens)

# Escolhe um token inicial aleatório para gerar o texto
start_token = random.choice(train_tokens)

# Gera um texto de 20 tokens usando o modelo de bigrama
generated_text = generate_text(bigram_model, start_token, num_tokens=20)

# Exibe o texto gerado
print("\nTexto gerado:")
print(generated_text)
# Exibindo o texto gerado na vertical
for word in generated_text.split():
    print(word)



Texto gerado:
# . estritamente > Recursos a A nº VLSI , Parpública No música : Um o o von || menor
#
.
estritamente
>
Recursos
a
A
nº
VLSI
,
Parpública
No
música
:
Um
o
o
von
||
menor


In [58]:
# Importar a função de cálculo de perplexidade
from bigram_model_perplexity import calculate_perplexity

# Exibindo cada palavra gerada na vertical e calculando a perplexidade após cada palavra
words = generated_text.split()

print("Palavra | Perplexidade")
print("-" * 30)

for i in range(2, len(words) + 1):
    # Considera o texto até a palavra atual
    partial_tokens = words[:i]
    
    # Calcula a perplexidade usando os tokens parciais
    perplexity = calculate_perplexity(bigram_model, partial_tokens)
    
    # Exibe a palavra atual e a perplexidade calculada
    print(f"{words[i - 1]} | {perplexity:.2f}")


Palavra | Perplexidade
------------------------------
. | 5.33
estritamente | 125.22
> | 72.41
Recursos | 106.35
a | 75.24
A | 87.59
nº | 135.40
VLSI | 147.85
, | 96.15
Parpública | 184.82
No | 119.63
música | 141.47
: | 136.73
Um | 170.78
o | 161.15
o | 152.58
von | 193.32
|| | 191.09
menor | 218.93


A perplexidade é uma métrica que indica o quão bem um modelo de linguagem prediz uma sequência de palavras. Em termos simples, a perplexidade mede a "surpresa" que o modelo tem ao prever a próxima palavra, com valores mais baixos indicando que o modelo faz previsões mais precisas.

### Como interpretar a perplexidade:
1. **Perplexidade baixa (boa)**:
   - Significa que o modelo é bom em prever a próxima palavra na sequência.
   - O modelo entende bem o padrão ou estrutura do texto.
2. **Perplexidade alta (ruim)**:
   - Indica que o modelo tem dificuldade em prever a próxima palavra.
   - O modelo não captura bem as dependências linguísticas.

### O que é uma "boa" perplexidade?
- Não há um valor fixo que defina uma "boa" ou "ruim" perplexidade, pois depende muito do tipo de modelo, do tamanho do vocabulário, e da complexidade do texto que você está usando.
- Em geral:
  - **Modelos simples**, como um bigrama, podem ter perplexidades relativamente altas, porque não capturam contextos complexos.
  - **Modelos mais avançados**, como aqueles baseados em redes neurais profundas, devem ter perplexidades significativamente mais baixas.

### Valores típicos:
- **Textos em inglês**: Perplexidades abaixo de 100 são frequentemente consideradas boas para modelos simples como unigramas ou bigramas. Para modelos mais complexos (como LSTM ou Transformers), valores bem menores são esperados.
- **Textos mais complexos**: Se o seu corpus é complicado ou técnico, uma perplexidade mais alta é esperada.

### Como saber se a perplexidade é boa no seu caso:
1. **Compare com outros modelos**: Teste a perplexidade de diferentes modelos (por exemplo, unigramas, bigramas, trigramas) e veja se a sua melhora.
2. **Contexto específico**: Se você tiver acesso a benchmarks ou valores de referência para o seu tipo de texto, compare com esses valores.
3. **Experimentos e ajustes**: Experimente ajustar o modelo ou os dados (como o pré-processamento) para ver se a perplexidade melhora.

### Exemplo prático:
- Se o seu modelo de bigrama tem uma perplexidade de 500, é um indicativo de que há bastante espaço para melhoria.
- Se a perplexidade é 50, o modelo está se saindo bem para um modelo simples de bigramas.

### Comparação da Perplexidade Global e Perplexidade por Palavra

#### Perplexidade Global
A perplexidade global que você obteve é **19,638.51**, o que é extremamente alto. Isso significa que, em média, o modelo está tendo muita dificuldade para prever as palavras subsequentes em relação ao conjunto de teste.

#### Perplexidade por Palavra
Vamos analisar as perplexidades individuais para algumas palavras:
- **Perplexidade baixa (ex.: "." com 5.33)**: Isso indica que o modelo tem mais confiança ou está mais certo sobre a previsão de um ponto final.
- **Perplexidade média (ex.: "a" com 75.24 e "Recursos" com 106.35)**: O modelo ainda está incerto, mas não completamente perdido em prever essas palavras.
- **Perplexidade alta (ex.: "Parpública" com 184.82 e "menor" com 218.93)**: Essas palavras são mais difíceis de prever para o modelo, talvez porque sejam mais raras ou porque o modelo de bigrama não consegue capturar bem o contexto necessário.

### O Que Isso Significa?
1. **Variação na perplexidade por palavra**: O modelo tem mais facilidade para prever palavras comuns ou aquelas que seguem padrões óbvios (como um ponto final), mas luta com palavras que são raras ou que exigem mais contexto, como "VLSI" e "Parpública".
2. **Perplexidade global muito alta**: Indica que, de forma geral, o modelo não está capturando bem os padrões linguísticos do texto. O valor médio de perplexidade individual é mais compreensível, mas mesmo assim relativamente alto.

### O Que Pode Ser Feito?
1. **Melhorar o modelo**:
   - Tente usar um modelo mais complexo, como trigrama ou redes neurais, que podem capturar melhor o contexto.
   - Considere aplicar técnicas de suavização (ex.: Laplace ou Kneser-Ney) para reduzir a perplexidade de palavras raras.

2. **Focar em palavras difíceis**:
   - Veja se há padrões específicos que o modelo está errando consistentemente. Talvez palavras como "VLSI" sejam termos técnicos que precisem de um tratamento especial.

3. **Analisar a distribuição do vocabulário**:
   - Verifique a frequência de palavras no corpus. Se houver muitas palavras raras, a perplexidade será naturalmente mais alta.

### Conclusão
- A perplexidade global muito alta sugere que o modelo de bigrama é limitado para o seu conjunto de dados.
- As perplexidades individuais mostram que o modelo pode ser razoável para palavras comuns, mas não é eficaz para palavras raras ou técnicas.